In [1]:
%pip install mcp python-dotenv
%pip install langchain-ollama


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


# Tests to verify the Local models are running fine.

In [2]:
# Test to see if the local llm is running and reachable.
from langchain_ollama import ChatOllama
local_model="qwen3:0.6b"
local_llm = ChatOllama(
    model = local_model, 
    # Dont' make things up.
    temperature = 0,
    stream = True,
)
#print(local_llm.invoke("How is the weather in Tehran right now?"))
# Make sure the model name matches exactly what you see in `ollama list`
for chunk in local_llm.stream("How is the weather in Tehran right now?"):
    print(chunk.content, end="", flush=True)

/home/rvv/Downloads/Voyager/voyager_env/lib/python3.14/site-packages/langchain_core/_api/deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


The current weather in Tehran is characterized by a dry and hot climate, with temperatures typically ranging from 40°C to 50°C. The air is dry, and there may be a slight breeze. It's important to note that Tehran experiences a continental climate, which makes it quite hot and dry throughout the day.

In [3]:
# Test to see if the local llm is running and reachable.
from langchain_ollama import ChatOllama
fast_model="smollm2:135m"
fast_local_llm = ChatOllama(
    model = fast_model, 
    # Dont' make things up.
    temperature = 0,
    stream = True,
)

# Make sure the model name matches exactly what you see in `ollama list`
for chunk in fast_local_llm.stream("How is the weather in Tehran right now?"):
    print(chunk.content, end="", flush=True)

The weather in Tehran is quite pleasant. The sun shines brightly and the air is filled with the sweet scent of blooming flowers. The temperature ranges from 25°C to 30°C (77°F to 86°F), making it perfect for outdoor activities like hiking, swimming, or simply enjoying a warm meal.

The weather forecast indicates that we are in a good period for the summer season, with temperatures ranging from 19°C to 25°C (64°F to 77°F). However, it's essential to note that this is still a relatively dry and hot region, so you might want to pack sunscreen or carry an umbrella if you plan on spending time outdoors.

If you're planning to visit Tehran in the summer, I recommend checking the local weather forecast before your trip to ensure you have enough time to acclimate to the heat.

In [ ]:
# News MCP

import asyncio
import ollama
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

# 1. Define your MCP Server (e.g., the Weather or News server)
server_params = StdioServerParameters(
    command="npx", 
    args=["-y", "@newsmcp/server"], # Example: News/Search
    env=None
)

async def run_ollama_mcp(user_prompt):
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            
            # 2. Get tools from MCP and format them for Ollama
            mcp_tools = await session.list_tools()
            ollama_tools = [
                {
                    "type": "function",
                    "function": {
                        "name": t.name,
                        "description": t.description,
                        "parameters": t.inputSchema,
                    },
                }
                for t in mcp_tools.tools
            ]
            # 3. Chat with Ollama
            messages = [
                {'role': 'system',
                 'content' : '''
                 You are a news curator. 
                  CRITICAL RULE: Always set the 'per_page' parameter to strictly 10
                  when calling the 'get_news' tool to save memory. 
                 
                 When presenting news:
                    1. Extract only the Title and a 1-sentence summary.
                    2. DO NOT include URLs, IDs, Impact scores, or Regions.
                    3. Use a clean bulleted list format."
                 You have two tools
                    1. `get_news`: Use this first to get a list of headlines and their UUIDs.
                    2. `get_news_detail`: Once you have a UUID from the list, you MUST call this tool to get the full story.

                    **Instructions:**
                    - When the user asks for news, call `get_news`.
                    - Look at the response. Find the UUID for the relevant article.
                    - Immediately call `get_news_detail` using that UUID.
                    - Do not ask the user for permission; just get the details and then summarize the final result.
                    
                 '''},
                {"role": "user", "content": user_prompt}]
            response = ollama.chat(
                model=local_model, # Make sure this model supports tools!
                messages=messages,
                tools=ollama_tools,
            )

            # 4. Handle Tool Calls
            if response.get('message', {}).get('tool_calls'):
                for tool_call in response['message']['tool_calls']:
                    # Execute the tool via the MCP Session
                    result = await session.call_tool(
                        tool_call['function']['name'], 
                        tool_call['function']['arguments']
                    )
                    # Format the MCP result into an Ollama tool response message
                    # We extract the text content from the MCP response
                    tool_output_text = " ".join([c.text for c in result.content if hasattr(c, 'text')])
                    print('--------------------')
                    print(f"Priting tool output {tool_output_text}")
                    print('--------------------')
                    # --- NEW: Curation step using fast local LLM to clean output ---
                    curation_prompt = f"""
                    Analyze the news content below and format it into a Risk Briefing.

                    REQUIRED FORMAT FOR EACH ITEM:
                    - **[Heading]** | **Verdict: [High Risk / Low Risk / No Risk]**
                    - Summary: [One sentence explaining the event]

                    RULES:
                    - No hypertext in the summary.
                    - Clean(Remove) all IDs, UUIDs, and URLs from the output.
                    - Do not mention the tools or \"get_news\".
                    - If no risk is apparent, mark as \"No Risk\".

                    CONTENT TO ANALYZE:
                    {tool_output_text}
                    """
                    curated_response = ollama.chat(
                        model=fast_model,
                        messages=[{'role': 'system', 'content': curation_prompt}]
                    )
                    curated_text = curated_response['message']['content']
                    print('--------------------')
                    print(f"Curation by small model {curated_text}")
                    print('--------------------')
                    print(f" Original text length {len(tool_output_text)}, Cureated text length {len(curated_text)}")
                    return curated_text
# Run in Jupyter
verdict = await run_ollama_mcp("Give me latest news from Iran.")
print(f"Final Verdict: {verdict}")

--------------------
Priting tool output # World News (5462 events, page 1/547)

Present the events below as a multi-story news briefing. Cover the top stories, not just one.

Formatting tip: Prioritize suppressing rich previews/cards. For Discord-style clients, output source URLs directly as '<https://...>' and avoid masked markdown links. Never emit raw standalone URLs outside no-preview wrappers.

## Sharif University of Technology, Iran's leading technical university, was bombed. It is unclear if the United States or Israel is responsible, though both have been implicated in recent attacks on Iranian universities. The attack on Sharif University, referred to as "Iran's MIT," targeted the information technology center and mosque, with reports suggesting some parts of the campus were used for drone research. 34 people were reportedly killed in bombings across Iran today, though the specific number at the university is unknown.
ID: beb83297-7758-4a92-8b2b-28b99e6b3004
Impact: 8 | Sour

In [11]:
print(verdict)

ID: ae7d75a4-fae0-428e-a923-a158ef16b488


Summary:
In the Middle East, tensions between Iran and Israel have escalated since President Donald Trump's announcement that he would invade Jerusalem in response to the 9/11 attacks. The situation is now particularly volatile due to recent attacks on critical infrastructure along the Gaza Strip border, including airstrikes targeting the US Embassy in Tel Aviv and an attack on the Israel Defense Forces' missile batteries stationed near the city.

The crisis has added significant humanitarian needs, as millions of people depend on critical infrastructure like hospitals, water treatment plants, and ports for their daily survival. The conflict is also causing delays in travel between major cities and affecting global trade.

Despite the heightened tensions, President Trump maintains that he would not invade Israel to prevent the attacks against the US embassy or other targets in the country. However, as of now, his stated stance means Iran could

In [ ]:
# Multi-MCP Trip Planner with Safety Logic
import ollama
import sys
from contextlib import AsyncExitStack
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

servers_config = {
    # "news": StdioServerParameters(command="npx", args=["-y", "@newsmcp/server"]),
    "weather": StdioServerParameters(command=sys.executable, args=["-m", "mcp_weather_server"])
}

async def run_multi_mcp_trip_planner():
    async with AsyncExitStack() as stack:
        sessions = {}
        all_ollama_tools = []
        tool_to_session = {}
        
        for name, params in servers_config.items():
            transport_read, transport_write = await stack.enter_async_context(stdio_client(params))
            session = await stack.enter_async_context(ClientSession(transport_read, transport_write))
            await session.initialize()
            sessions[name] = session
            
            mcp_tools = await session.list_tools()
            for t in mcp_tools.tools:
                tool_to_session[t.name] = session
                all_ollama_tools.append({
                    "type": "function",
                    "function": {
                        "name": t.name,
                        "description": t.description,
                        "parameters": t.inputSchema,
                    },
                })

        # Use the global news_summary from the previous cell
        safety_context = f"Latest News for context:\n{verdict}"
        
        messages = [
            {'role': 'system', 'content': f'''You are a safety-first travel planner.
            CONTEXT:
            {safety_context}
            
            CRITICAL RULES:
            1. Analyze the News and Weather carefully.
            2. If there are ADVERSE conditions (storms, social unrest, safety alerts), DO NOT plan a trip to that location.
            3. Instead, EXPLAIN the risks and recommend a safer alternative destination.
            4. If conditions are good, provide a 3-day itinerary.'''},
            {'role': 'user', 'content': "Check the weather and latest news from and decide if I should plan a trip to Iran right now. If it's safe, give me a plan. If not, suggest somewhere else."}
        ]

        while True:
            response = ollama.chat(
                model=local_model, 
                messages=messages,
                tools=all_ollama_tools,
            )
            
            if not response.get('message', {}).get('tool_calls'):
                messages.append(response['message'])
                break

            messages.append(response['message'])
            for tool_call in response['message']['tool_calls']:
                tool_name = tool_call['function']['name']
                session = tool_to_session.get(tool_name)
                result = await session.call_tool(tool_name, tool_call['function']['arguments'])
                result_content = " ".join([c.text for c in result.content if hasattr(c, 'text')])
                
                messages.append({
                    'role': 'tool',
                    'content': result_content,
                    'tool_call_id': tool_call.get('id') or tool_call.get('function', {}).get('id', 'default_id')
                })
        
        print("\n--- SAFETY EVALUATION & TRIP PLAN ---")
        print(messages[-1]['content'])

await run_multi_mcp_trip_planner()
